# DSA 210 Analysis Notebook\n## Impact of Content Consistency on Social Media Growth\n\n**Student:** Zainab Siddiqui (36082)\n\n**Note:** This notebook currently uses a realistic synthetic dataset that matches the proposed project structure. It demonstrates the required workflow for the 14 April milestone: data collection structure, EDA, and hypothesis testing.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.stats import pearsonr, ttest_ind\nimport statsmodels.api as sm\n\ndf = pd.read_csv('../data/youtube_consistency_dataset.csv')\ncreator_summary = pd.read_csv('../data/creator_summary.csv')\n\ndf.head()

## 1. Basic structure and descriptive statistics

In [ ]:
print('Rows, columns:', df.shape)\nprint('Creators:', df['creator'].nunique())\nprint('Date range:', df['upload_date'].min(), 'to', df['upload_date'].max())\ndf[['views','likes','comments','duration_min','upload_interval_days','engagement_rate']].describe()

## 2. Creator-level summary

In [ ]:
creator_summary

## 3. EDA

In [ ]:
# Plot 1: Average views by creator\navg_views = df.groupby('creator')['views'].mean().sort_values()\nplt.figure(figsize=(9,5))\navg_views.plot(kind='barh')\nplt.title('Average Views by Creator')\nplt.xlabel('Average views')\nplt.tight_layout()\nplt.show()

In [ ]:
# Plot 2: Consistency index vs average views\nplt.figure(figsize=(7,5))\nplt.scatter(creator_summary['consistency_index'], creator_summary['avg_views'])\nfor _, row in creator_summary.iterrows():\n    plt.annotate(row['creator'], (row['consistency_index'], row['avg_views']))\nplt.title('Consistency Index vs Average Views')\nplt.xlabel('Consistency index')\nplt.ylabel('Average views')\nplt.tight_layout()\nplt.show()

In [ ]:
# Plot 3: Upload interval distribution\nplt.figure(figsize=(7,5))\ndf['upload_interval_days'].dropna().hist(bins=20)\nplt.title('Distribution of Upload Intervals')\nplt.xlabel('Days between uploads')\nplt.ylabel('Frequency')\nplt.tight_layout()\nplt.show()

In [ ]:
# Plot 4: Engagement by consistency group\ngrouped = df.groupby('consistency_group')['engagement_rate'].mean()\nplt.figure(figsize=(6,4))\ngrouped.plot(kind='bar')\nplt.title('Average Engagement Rate by Consistency Group')\nplt.ylabel('Average engagement rate')\nplt.tight_layout()\nplt.show()

## 4. Hypothesis testing

In [ ]:
# Hypothesis 1: Correlation between consistency index and average views\ncorr, p_value = pearsonr(creator_summary['consistency_index'], creator_summary['avg_views'])\nprint('Pearson correlation:', round(corr, 4))\nprint('p-value:', round(p_value, 6))

In [ ]:
# Hypothesis 2: T-test for engagement by consistency group\nhigh = df[df['consistency_group']=='High consistency']['engagement_rate']\nlow = df[df['consistency_group']=='Low consistency']['engagement_rate']\nt_stat, p_val = ttest_ind(high, low, equal_var=False)\nprint('T-statistic:', round(t_stat, 4))\nprint('p-value:', round(p_val, 6))

In [ ]:
# Hypothesis 3: Regression model\nreg_df = df.dropna(subset=['upload_interval_days']).copy()\nX = reg_df[['consistency_index', 'duration_min', 'subscriber_proxy', 'upload_interval_days']]\nX = sm.add_constant(X)\ny = reg_df['views']\nmodel = sm.OLS(y, X).fit()\nprint(model.summary())

## 5. Interpretation

- If the correlation is positive, more consistent creators tend to have higher average views.\n- If the t-test p-value is below 0.05, engagement differs significantly between the two consistency groups.\n- In the regression, positive coefficients for `consistency_index` suggest that regular posting is associated with stronger performance after controlling for other variables.